In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/IS3107/combined_v4.csv')
print(df.shape)
pd.set_option('display.max_columns', 100)
df.head(3)

# Data prep

In [ ]:
df_clean = df.copy()

## Handling NaNs

In [ ]:
# get rid of cols that are irrelevant to salary prediction
df_clean = df_clean.drop(['Created_date', 'Job_id', 'Founded', 'Revenue', 'Application_link'], axis=1) # revenue and founded corresponds to the company anyway, not needed

In [ ]:
# initial nan count
print(df_clean.isna().sum())

In [ ]:
# we cannot make salary predictions on jobs that don't even salary data. hence, drop rows without salary data
df_clean = df_clean.dropna(subset=['Salary_min_month'])

# drop original salary col
df_clean = df_clean.drop(['Salary'], axis=1)
print(df_clean.shape)

In [ ]:
# handle nans in non-numeric cols
non_numeric_cols = ['Company', 'Requirements_short', 'Requirements_full', 'Type',
       'Responsibilities', 'Location', 'Industry', 'Sector', 'Availability_requests', 'Size']
for i in non_numeric_cols:
  df_clean[i] = df_clean[i].fillna('Unknown') # since the information is unknown, we should literally fill nans up with "Unknown"

# handling 'Field' -- since it only has 21 nans, we will impute with the most common title, which is software engineer
df_clean['Field'] = df_clean['Field'].fillna('Software engineer')

In [ ]:
# drop Title, bcos Field already contains the parsed form
df_clean = df_clean.drop(['Title'], axis=1)

In [ ]:
# handle nans in numeric cols
df_clean['Is_internship'] = df_clean['Is_internship'].fillna(0)

In [ ]:
# final nan count
print(df_clean.isna().sum())

## Handling dtypes

In [ ]:
# initial dtypes
print(df_clean.dtypes)

In [ ]:
df_clean2 = df_clean.copy()

In [ ]:
# Drop Data_source -- irrelevant
df_clean2 = df_clean2.drop(['Data_source'], axis=1)
df_clean2 = df_clean2.drop(['Unnamed: 0'], axis=1)

In [ ]:
df_clean2['Date_scraped'] = pd.to_datetime(df_clean2['Date_scraped'])
df_clean2['Field'] = df_clean2['Field'].str.replace("[^a-zA-Z\s]", "", regex=True)

In [ ]:
# since description is derived from requirements, drop requirements col
df_clean2 = df_clean2.drop(['Requirements_short', 'Requirements_full'], axis=1)

## handling categorical variables

In [ ]:
# checking which cols are categorical
print(df_clean2.nunique())

In [ ]:
# binning for categorical variables

# checking current value counts for these categorical variables
cat = ['Type', 'Location', 'Size', 'Sector', 'Availability_requests', 'Industry', 'Company'] # we will handle 'Field' separately from the rest due to different preprocessing needed

for i in cat:
  print(f'Value counts for column: {i}')
  print(df_clean2[i].value_counts())
  print('*'*50)

In [ ]:
# handling the binning

# for highly imbalanced categories, it is not meaningful to keep these columns, since the values should be the same for almost the whole dataset
# in some cases, e.g. Location, it is indeed the same Location for the whole dataset (Singapore). But due to recording differences between different platforms, there are different categories showing up

# drop these categories
df_clean2 = df_clean2.drop(['Availability_requests', 'Sector', 'Location', 'Size', 'Industry', 'Company'], axis=1)

In [ ]:
# re-binning for 'Type'
df_clean2['Job_Type'] = np.where(
    df_clean2['Type'].str.contains('Full'), 'Full time',
      np.where(df_clean2['Type'].str.contains('Perm'), 'Contract',
        np.where(df_clean2['Type'].str.contains('Unknown'), 'Unknown',
          np.where(df_clean2['Type'].str.contains('Fresh graduate'), 'Full time',
            'Contract'))))

# drop original 'Type' column
df_clean2 = df_clean2.drop(['Type'], axis=1)

# checking final Job_Type binning
print(df_clean2['Job_Type'].value_counts())

In [ ]:
# handling 'Field'
print(df_clean2['Field'].value_counts())

In [ ]:
df_clean2['Job_Field'] = np.where(
    df_clean2['Field'].str.contains('analyst', case=False), 'data analyst',
      np.where(df_clean2['Field'].str.contains('scientist', case=False), 'data scientist',
        np.where((df_clean2['Field'].str.contains('data engineer', case=False)) | (df_clean2['Field'].str.contains('machine learning engineer', case=False)), 'data engineer',
          np.where(df_clean2['Field'].str.contains('architect', case=False), 'data architect',
            np.where(df_clean2['Field'].str.contains('database', case=False), 'database administrator',
              np.where((df_clean2['Field'].str.contains('software engineer', case=False)) | (df_clean2['Field'].str.contains('developer', case=False)), 'software engineer', 'Others')
                # while 'Others' may seem very vague, from our list of filtered keywords, we know that these are all still data-related jobs.
)))))

# drop original 'Field' column
df_clean2 = df_clean2.drop(['Field'], axis=1)

# checking final Job_Field binning
print(df_clean2['Job_Field'].value_counts())

In [ ]:
# handling 'Description'

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Vectorize the Text
vectorizer = CountVectorizer(stop_words='english')
dtm = vectorizer.fit_transform(df_clean2['Description'])  # Document-term matrix

# Fit LDA Model
lda = LatentDirichletAllocation(n_components=10, random_state=0)  # Assuming we want to find 10 topics
lda_features = lda.fit_transform(dtm)

# Print the topics
def print_topics(model, vectorizer, top_n=10):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx + 1))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
               for i in topic.argsort()[:-top_n - 1:-1]])
print_topics(lda, vectorizer)

In [ ]:
# just take the highest probability topic as a feature for simplicity
df_clean2['Dominant_Topic'] = np.argmax(lda_features, axis=1)

# drop original Description column
df_clean2 = df_clean2.drop(['Description'], axis=1)

In [ ]:
# drop Responsibities
df_clean2 = df_clean2.drop(['Responsibilities'], axis=1)

In [ ]:
# create a new salary column, which is the midpoint of salary_min and salary_max, so that we only have 1 variable to predict
df_clean2['Salary_midpoint'] = (df_clean2['Salary_min_month'] + df_clean2['Salary_max_month']) / 2

# drop original salary columns
df_clean2 = df_clean2.drop(['Salary_min_month', 'Salary_max_month'], axis=1)
df_clean2.head(3)

In [ ]:
# drop date column -- not doing time series
df_clean2 = df_clean2.drop(['Date_scraped'], axis=1)

In [ ]:
df_clean2['Is_internship'] = df_clean2['Is_internship'].astype('int')

In [ ]:
df_clean2.head()

In [ ]:
df_clean2.corr()

In [ ]:
import seaborn as sns
plt.figure(figsize=(5, 2))
sns.boxplot(data=df_clean2, x='Salary_midpoint')

# Machine Learning :D

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df_clean2['Job_Type'] = label_encoder.fit_transform(df_clean2['Job_Type'])
df_clean2['Job_Field'] = label_encoder.fit_transform(df_clean2['Job_Field'])

In [ ]:
# scaling target
from sklearn.preprocessing import MinMaxScaler

X = df_clean2.drop(['Salary_midpoint'], axis=1)
y = df_clean2['Salary_midpoint']

scaler = MinMaxScaler(feature_range=(0, 1))
y = scaler.fit_transform(y.values.reshape(-1, 1))

In [ ]:
# train test split -- no need val since we are only using 1 model and our dataset so small
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05, random_state=42)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

random_forest_regressor = RandomForestRegressor(
                              n_estimators=500,
                              max_depth=None,
                              max_features='sqrt',
                              random_state=42)
random_forest_regressor.fit(X_train, y_train)

In [ ]:
# Make predictions for test set
y_pred = random_forest_regressor.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)
r2 = r2_score(y_test, y_pred)
print("Test R squared:", r2)

In [ ]:
# Evaluation on train seet
y_pred_train = random_forest_regressor.predict(X_train)

# Evaluate the model
mse = mean_squared_error(y_train, y_pred_train)
print("Mean Squared Error:", mse)
r2 = r2_score(y_train, y_pred_train)
print("Train R squared:", r2)

In [ ]:
import matplotlib.pyplot as plt

# Actual vs Predicted scatter plot
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7, color='blue', edgecolors='w', label='Predicted vs Actual')
plt.title('Actual vs Predicted Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)  # Diagonal line
plt.legend()
plt.show()

# Remove outliers

In [ ]:
df_clean2.head()

In [ ]:
Q1 = df_clean2['Salary_midpoint'].quantile(0.25)
Q3 = df_clean2['Salary_midpoint'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean_no_outliers = df_clean2[(df_clean2['Salary_midpoint'] >= lower_bound) & (df_clean2['Salary_midpoint'] <= upper_bound)]

print(df_clean_no_outliers.shape)
df_clean_no_outliers.head()

In [ ]:
# scaling target
from sklearn.preprocessing import MinMaxScaler

X = df_clean_no_outliers.drop(['Salary_midpoint'], axis=1)
y = df_clean_no_outliers['Salary_midpoint']

scaler = MinMaxScaler(feature_range=(0, 1))
y = scaler.fit_transform(y.values.reshape(-1, 1))

In [ ]:
# train test split -- no need val since we are only using 1 model and our dataset so small
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05, random_state=42)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

random_forest_regressor = RandomForestRegressor(
                              n_estimators=700,
                              max_depth=10,
                              max_features='log2',
                              warm_start=True,
                              random_state=42)
random_forest_regressor.fit(X_train, y_train)

In [ ]:
# Make predictions for test set
y_pred = random_forest_regressor.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)
r2 = r2_score(y_test, y_pred)
print("Test R squared:", r2)

print('*'*50)

# Evaluation on train seet
y_pred_train = random_forest_regressor.predict(X_train)

# Evaluate the model
mse = mean_squared_error(y_train, y_pred_train)
print("Mean Squared Error:", mse)
r2 = r2_score(y_train, y_pred_train)
print("Train R squared:", r2)

In [ ]:
import matplotlib.pyplot as plt

# Actual vs Predicted scatter plot
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7, color='blue', edgecolors='w', label='Predicted vs Actual')
plt.title('Actual vs Predicted Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)  # Diagonal line
plt.legend()
plt.show()